# 6. Model Comparison & Final Evaluation

This notebook compares all three models side by side:
- **Logistic Regression** (baseline)
- **ANN** (deep learning)
- **Random Forest** (ensemble)

We compare metrics, overlay ROC curves, and present confusion matrices for the final assessment.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle

sns.set_style('whitegrid')

In [2]:
# Load preprocessed test data
data = np.load('../data/preprocessed_data.npz')
X_test = data['X_test']
y_test = data['y_test']
feature_names = np.load('../data/feature_names.npy', allow_pickle=True)

print(f'Test set: {X_test.shape}')

Test set: (2466, 17)


In [3]:
# Load all three trained models
save_dir = Path('../data')

lr_model = joblib.load(save_dir / 'lr_model.joblib')

import tensorflow as tf
ann_model = tf.keras.models.load_model(save_dir / 'ann_model.keras')

rf_model = joblib.load(save_dir / 'rf_model.joblib')

print('All 3 models loaded successfully.')

All 3 models loaded successfully.


In [4]:
from src.evaluation import compute_metrics, plot_comparative_roc
from sklearn.metrics import confusion_matrix

## 6.1 Metrics Comparison Table

In [5]:
# Compute metrics for all models
models = {
    'Logistic Regression': lr_model,
    'ANN': ann_model,
    'Random Forest': rf_model,
}

all_results = {}
for name, model in models.items():
    all_results[name] = compute_metrics(model, X_test, y_test, name)

# Build comparison DataFrame
comparison_data = []
for name, res in all_results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall': f"{res['recall']:.4f}",
        'F1-Score': f"{res['f1']:.4f}",
        'ROC-AUC': f"{res['roc_auc']:.4f}",
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')
print('=' * 70)
print('MODEL COMPARISON')
print('=' * 70)
comparison_df

MODEL COMPARISON


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Model,,,,,
Logistic Regression,0.8593,0.5348,0.7042,0.6079,0.8801
ANN,0.8528,0.5174,0.7408,0.6093,0.8946
Random Forest,0.8897,0.6297,0.6990,0.6625,0.9217


## 6.2 Comparative ROC Curves

The ROC curve plots the True Positive Rate against the False Positive Rate at various threshold settings. A higher AUC indicates better discriminative ability.

In [6]:
# Prepare data for comparative ROC
roc_data = {}
for name, res in all_results.items():
    roc_data[name] = {
        'y_true': y_test,
        'y_proba': res['y_proba'],
    }

plot_comparative_roc(roc_data, save_path='../report/Resources/comparative_roc.pdf')

## 6.3 Confusion Matrices Side by Side

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, all_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No Purchase', 'Purchase'],
        yticklabels=['No Purchase', 'Purchase'],
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name}\n(F1={res["f1"]:.3f}, AUC={res["roc_auc"]:.3f})')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../report/Resources/comparative_confusion_matrices.pdf', format='pdf', bbox_inches='tight')
plt.show()

## 6.4 Analysis & Discussion

### Key Observations

1. **ROC-AUC**: Measures the model's ability to discriminate between purchasers and non-purchasers across all thresholds. Higher is better.

2. **F1-Score**: The harmonic mean of Precision and Recall at the default 0.5 threshold. Important for operational deployment where a specific threshold must be chosen.

3. **Precision vs. Recall Trade-off**: 
   - High **precision** → fewer false alarms (predicting purchase when there isn't one)
   - High **recall** → fewer missed purchases (failing to identify actual buyers)
   - The optimal balance depends on business objectives

4. **Model Complexity vs. Performance**: 
   - Logistic Regression is the most interpretable but may underfit non-linear patterns
   - The ANN can capture complex interactions but is a "black box"
   - Random Forest combines strong performance with SHAP interpretability

### Why Both F1 and ROC-AUC?

- F1 evaluates at a **specific decision threshold** — relevant for deployment
- ROC-AUC evaluates across **all thresholds** — relevant for model comparison
- A model can have high AUC but mediocre F1 if the default threshold isn't optimal

## 6.5 Conclusions

### Final Model Comparison

All three models were trained on SMOTE-resampled training data and evaluated on the original (imbalanced) test set to ensure realistic performance estimates.

**Recommendations**:
- **For interpretability**: Use Logistic Regression — coefficients directly explain feature influence
- **For best predictive performance**: The model with the highest ROC-AUC and F1 combination should be preferred
- **For interpretable + accurate**: Random Forest with SHAP explanations provides the best of both worlds

### Reproducibility

All experiments use `random_state=42`. Run notebooks 01→06 sequentially to reproduce all results.